In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor

from sklearn.multioutput import MultiOutputRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

Pandas to load dataset

OneHotEncoder to Convert text into numbers

RandomForest to Learn nutrition patterns

Metrics to Evaluate model

Joblib to Save trained model

In [2]:
df = pd.read_csv("../data/processed/training_dataset.csv")

In [3]:
df.head()

,age,gender,height_m,weight_kg,bmi,health_condition,activity_level,diet_type,allergy,target_protein,...,target_fiber,target_max_sugar,target_max_sodium,target_calories,target_protein.1,target_carbohydrates.1,target_fat.1,target_fiber.1,target_max_sugar.1,target_max_sodium.1
0,71,Female,1.61,112,26.5,Diabetes,Low,Non-Vegetarian,NaN,90,...,35,25,1800,1400,90,155,55,35,25,1800
1,56,Female,1.79,111,19.1,Healthy,Medium,Non-Vegetarian,NaN,75,...,30,50,2300,2200,75,275,70,30,50,2300
2,34,Female,1.72,96,19.3,Healthy,Low,Vegan,Dairy,75,...,30,50,2300,2000,75,260,70,30,50,2300
3,79,Female,1.68,114,25.9,Healthy,Medium,Vegetarian,Dairy,75,...,30,50,2300,1950,75,265,70,30,50,2300
4,67,Male,1.65,47,19.8,Hypertension,Low,Non-Vegetarian,Seafood,80,...,35,40,1500,1650,80,225,60,35,40,1500


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   age                     20000 non-null  int64  
 1   gender                  20000 non-null  str    
 2   height_m                20000 non-null  float64
 3   weight_kg               20000 non-null  int64  
 4   bmi                     20000 non-null  float64
 5   health_condition        20000 non-null  str    
 6   activity_level          20000 non-null  str    
 7   diet_type               20000 non-null  str    
 8   allergy                 5038 non-null   str    
 9   target_protein          20000 non-null  int64  
 10  target_carbohydrates    20000 non-null  int64  
 11  target_fat              20000 non-null  int64  
 12  target_fiber            20000 non-null  int64  
 13  target_max_sugar        20000 non-null  int64  
 14  target_max_sodium       20000 non-null  int64  
 

In [5]:
df.isnull().sum()

age                           0
gender                        0
height_m                      0
weight_kg                     0
bmi                           0
health_condition              0
activity_level                0
diet_type                     0
allergy                   14962
target_protein                0
target_carbohydrates          0
target_fat                    0
target_fiber                  0
target_max_sugar              0
target_max_sodium             0
target_calories               0
target_protein.1              0
target_carbohydrates.1        0
target_fat.1                  0
target_fiber.1                0
target_max_sugar.1            0
target_max_sodium.1           0
dtype: int64

Find the null values in training_dataset and handle them.

Because, ML will not understand the null values in training process.

it's better to explicitly say, No Allergy

In [6]:
df["allergy"] = df["allergy"].fillna("No Allergy")

In [7]:
df["allergy"].value_counts()

allergy
No Allergy    14962
Dairy          1617
Gluten         1182
Peanut         1110
Seafood         750
Soy             379
Name: count, dtype: int64

first merged nutrition_rules.csv, DataFrame already had:

target_protein

target_carbohydrates

target_fat

target_fiber

target_max_sugar

target_max_sodium

Later, adjust_nutrition_targets() returned these same column names again, so when Pandas concatenated them, it automatically created:

target_protein.1

target_carbohydrates.1


The .1 columns contain the final adjusted values and are the ones we want to train on.

In [8]:
print(df.columns.tolist())

['age', 'gender', 'height_m', 'weight_kg', 'bmi', 'health_condition', 'activity_level', 'diet_type', 'allergy', 'target_protein', 'target_carbohydrates', 'target_fat', 'target_fiber', 'target_max_sugar', 'target_max_sodium', 'target_calories', 'target_protein.1', 'target_carbohydrates.1', 'target_fat.1', 'target_fiber.1', 'target_max_sugar.1', 'target_max_sodium.1']


Remove the old columns which are in nutrition_rules.csv

In [9]:
df = df.drop(columns=[
    "target_protein",
    "target_carbohydrates",
    "target_fat",
    "target_fiber",
    "target_max_sugar",
    "target_max_sodium"
])

Rename the New Ones

In [10]:
df = df.rename(columns={
    "target_protein.1": "target_protein",
    "target_carbohydrates.1": "target_carbohydrates",
    "target_fat.1": "target_fat",
    "target_fiber.1": "target_fiber",
    "target_max_sugar.1": "target_max_sugar",
    "target_max_sodium.1": "target_max_sodium"
})

In [11]:
df["allergy"] = df["allergy"].fillna("No Allergy")

In [12]:
df["allergy"].value_counts()

allergy
No Allergy    14962
Dairy          1617
Gluten         1182
Peanut         1110
Seafood         750
Soy             379
Name: count, dtype: int64

In [13]:
print(df.columns.tolist())

['age', 'gender', 'height_m', 'weight_kg', 'bmi', 'health_condition', 'activity_level', 'diet_type', 'allergy', 'target_calories', 'target_protein', 'target_carbohydrates', 'target_fat', 'target_fiber', 'target_max_sugar', 'target_max_sodium']


In [14]:
df.to_csv(
    "../data/processed/training_dataset.csv",
    index=False
)